# HW_5

### Task 1

In [4]:
import os
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from tqdm.auto import tqdm
from bert_score import score as bert_score
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForQuestionAnswering
from huggingface_hub import snapshot_download


In [7]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CACHE_DIR = Path.home() / '.cache' / 'huggingface' / 'datasets'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

HF_HOME = Path.home() / '.cache' / 'huggingface'
HF_HOME.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SAMPLES = 1000
MAX_NEW_TOKENS = 12
BERTSCORE_MODEL = 'distilroberta-base'
SAMPLE_CACHE_PATH = Path('HW_5/cache/mirage_sample_1000.pkl')
SAMPLE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

MIRAGE_DIR = Path.home() / '.cache' / 'huggingface' / 'hub' / 'datasets--nlpai-lab--mirage'
MIRAGE_REV = (MIRAGE_DIR / 'refs' / 'main').read_text().strip()
MIRAGE_TRAIN_PATH = MIRAGE_DIR / 'snapshots' / MIRAGE_REV / 'data' / 'train-00000-of-00001.parquet'


In [14]:
ds = load_dataset(
    'parquet',
    data_files=str(MIRAGE_TRAIN_PATH),
    split='train',
    cache_dir=str(CACHE_DIR),
)

def to_answers(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [str(a) for a in x if str(a).strip()]
    return [str(x)] if str(x).strip() else []

df = ds.to_pandas()[['query_id', 'query', 'answer', 'oracle']].copy()
df['answers'] = df['answer'].apply(to_answers)
df = df.rename(columns={'query': 'question'})

required_cols = {'query_id', 'question', 'answers', 'oracle'}
if SAMPLE_CACHE_PATH.exists():
    cached_df = pd.read_pickle(SAMPLE_CACHE_PATH)
    if required_cols.issubset(set(cached_df.columns)):
        df = cached_df
        print(f'Loaded cached sample: {SAMPLE_CACHE_PATH}')
    else:
        print('Cached sample is outdated. Rebuilding cache...')
        SAMPLE_CACHE_PATH.unlink(missing_ok=True)
        if MAX_SAMPLES is not None:
            df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
        df.to_pickle(SAMPLE_CACHE_PATH)
        print(f'Saved cached sample: {SAMPLE_CACHE_PATH}')
else:
    if MAX_SAMPLES is not None:
        df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
    df.to_pickle(SAMPLE_CACHE_PATH)
    print(f'Saved cached sample: {SAMPLE_CACHE_PATH}')

df.shape

Cached sample is outdated. Rebuilding cache...
Saved cached sample: HW_5/cache/mirage_sample_1000.pkl


(1000, 5)

In [20]:
_ws_re = re.compile(r'\s+')
_punct_re = re.compile(r'[^\w\s]')

def normalize_text(text: str) -> str:
    s = str(text).lower()
    s = _punct_re.sub(' ', s)
    s = _ws_re.sub(' ', s).strip()
    return s

def em_squad(pred: str, refs: list[str]) -> float:
    p = normalize_text(pred)
    return float(any(p == normalize_text(r) for r in refs))

def f1_pair(pred: str, ref: str) -> float:
    p_tokens = normalize_text(pred).split()
    r_tokens = normalize_text(ref).split()
    if not p_tokens and not r_tokens:
        return 1.0
    if not p_tokens or not r_tokens:
        return 0.0
    common = Counter(p_tokens) & Counter(r_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p_tokens)
    recall = overlap / len(r_tokens)
    return 2 * precision * recall / (precision + recall)

def f1_squad(pred: str, refs: list[str]) -> float:
    return max((f1_pair(pred, r) for r in refs), default=0.0)

def em_loose_mirage(pred: str, refs: list[str]) -> float:
    p = normalize_text(pred)
    if not p:
        return 0.0
    for r in refs:
        rr = normalize_text(r)
        if rr and (p in rr or rr in p):
            return 1.0
    return 0.0

def em_strict_mirage(pred: str, refs: list[str]) -> float:
    return em_squad(pred, refs)


In [27]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    local_files_only=True,
)

def generate_0shot_answer(question: str) -> str:
    messages = [
        {
            'role': 'system',
            'content': 'You answer factoid questions with a short factoid answer only. Do not write a full sentence.',
        },
        {
            'role': 'user',
            'content': f'Question: {question}\nGive only the answer phrase. Do not write a full sentence.',
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    ans = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip().split('\n')[0].strip()
    if '.' in ans:
        ans = ans.split('.')[0].strip()
    return ans


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2870.13it/s]


### Smoke-test (1 вопрос)

In [28]:
q = df.iloc[0]['question']
print('Q:', q)
print('A:', generate_0shot_answer(q))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: the school of gymnastics is called in athens as
A: Athens' Gymnastics School


### Прогон

In [49]:
rows = []
for r in tqdm(df.itertuples(index=False), total=len(df), desc='Task1 0-shot'):
    pred = generate_0shot_answer(r.question)
    rows.append({
        'query_id': r.query_id,
        'question': r.question,
        'answers': r.answers,
        'prediction': pred,
    })
pred_df = pd.DataFrame(rows)
pred_df.head(5)


Task1 0-shot: 100%|██████████| 1000/1000 [12:31<00:00,  1.33it/s]


,query_id,question,answers,prediction
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]",Athens' Gymnastics School
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",South Carolina
2,742f34d8-c3ed-49b1-acbe-455378b1f328,In what city was Joachim Olsen born?,"[Aalborg, Ålborg]","Oslo, Norway"
3,cb036dcf-6d57-40dd-b0af-f97cf9b057fa,Who is the father of Charites?,[Zeus],St Augustine
4,76212448-e12d-4f2c-8bb6-d944d033af85,Who was the composer of Stormy Monday?,"[Mike Figgis, Michael Lawrence Dundus Figgis, ...",John Williams


In [50]:
pred_df.to_pickle("HW_5_task1_predictions.pkl")

In [51]:
em_vals, f1_vals, loose_vals, strict_vals = [], [], [], []

for r in pred_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    em_vals.append(em_squad(r.prediction, refs))
    f1_vals.append(f1_squad(r.prediction, refs))
    loose_vals.append(em_loose_mirage(r.prediction, refs))
    strict_vals.append(em_strict_mirage(r.prediction, refs))

best_refs = []
for r in pred_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    best_refs.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else '')

P, R, F = bert_score(
    pred_df['prediction'].astype(str).tolist(),
    best_refs,
    model_type=BERTSCORE_MODEL,
    lang='en',
    verbose=False,
)

metrics = {
    'EM': float(np.mean(em_vals)),
    'F1': float(np.mean(f1_vals)),
    'EM_loose': float(np.mean(loose_vals)),
    'EM_strict': float(np.mean(strict_vals)),
    'BERTScore_F1': float(F.mean().item()),
    'N': int(len(pred_df)),
}
pd.DataFrame([metrics])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6492.73it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,0.01,0.057022,0.041,0.01,0.863311,1000


In [52]:
pred_df.to_csv('HW_5_task1_predictions.csv', index=False)
pd.DataFrame([metrics]).to_csv('HW_5_task1_metrics.csv', index=False)
print('Saved: HW_5_task1_predictions.csv, HW_5_task1_metrics.csv')


Saved: HW_5_task1_predictions.csv, HW_5_task1_metrics.csv


### вывод
- На выборке `N=1000` получены: `EM=0.010`, `F1=0.057`, `EM_loose=0.041`, `EM_strict=0.010`, `BERTScore_F1=0.863`.
- По строгим метрикам (`EM`, `EM_strict`) качество низкое: точное совпадение встречается редко.
- При этом `BERTScore_F1` заметно выше, что указывает на частичную семантическую близость ответов при слабом лексическом совпадении.


## Task 2

In [2]:
QA_MODEL_ID = "deepset/roberta-base-squad2"
TASK2_PRED_PATH = Path("HW_5_task2_predictions.csv")
TASK2_METRICS_PATH = Path("HW_5_task2_metrics.csv")

In [ ]:
# _ = snapshot_download(repo_id=QA_MODEL_ID, repo_type="model")
# print("Model cached:", QA_MODEL_ID)

Fetching 12 files: 100%|██████████| 12/12 [05:03<00:00, 25.30s/it]

Model cached: deepset/roberta-base-squad2


In [5]:
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_ID, local_files_only=True)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_ID, local_files_only=True)
qa_model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7716.82it/s]
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForQuestionAnswering(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (Lay

In [15]:
task2_df = df.copy()
task2_df["oracle_passage"] = task2_df["oracle"].apply(
    lambda x: x.get("doc_chunk", "") if isinstance(x, dict) else ""
)

# Оставляем нужные поля для инференса
task2_df = task2_df[["query_id", "question", "answers", "oracle_passage"]]
task2_df.head(2)

,query_id,question,answers,oracle_passage
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]","Accordingly, the gymnasium became connected wi..."
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",Intercollegiate sports teams of Duke Universit...


In [16]:
def extractive_answer(question: str, context: str, max_length: int = 512) -> str:
    if not isinstance(context, str) or not context.strip():
        return ""

    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=max_length,
    )

    with torch.no_grad():
        out = qa_model(**inputs)

    start = int(out.start_logits.argmax())
    end = int(out.end_logits.argmax())
    if end < start:
        end = start

    ans_ids = inputs["input_ids"][0][start:end + 1]
    ans = qa_tokenizer.decode(ans_ids, skip_special_tokens=True).strip()
    return ans


In [17]:
rows2 = []
for r in tqdm(task2_df.itertuples(index=False), total=len(task2_df), desc="Task2 extractive QA"):
    pred = extractive_answer(r.question, r.oracle_passage)
    rows2.append({
        "query_id": r.query_id,
        "question": r.question,
        "answers": r.answers,
        "oracle_passage": r.oracle_passage,
        "prediction": pred,
    })

pred2_df = pd.DataFrame(rows2)
pred2_df.head(5)


Task2 extractive QA: 100%|██████████| 1000/1000 [01:49<00:00,  9.10it/s]


,query_id,question,answers,oracle_passage,prediction
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]","Accordingly, the gymnasium became connected wi...",the Academy
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",Intercollegiate sports teams of Duke Universit...,"les Diables Bleus"" or ""the Blue Devils,"""
2,742f34d8-c3ed-49b1-acbe-455378b1f328,In what city was Joachim Olsen born?,"[Aalborg, Ålborg]",Danish politician and shot putter\nJoachim Brø...,Aalborg
3,cb036dcf-6d57-40dd-b0af-f97cf9b057fa,Who is the father of Charites?,[Zeus],Greek goddesses of grace and beauty\nIn Greek ...,
4,76212448-e12d-4f2c-8bb6-d944d033af85,Who was the composer of Stormy Monday?,"[Mike Figgis, Michael Lawrence Dundus Figgis, ...",1988 crime thriller film by Mike Figgis\nStorm...,Who was the composer of Stormy Monday?1988 cri...


In [22]:
em_vals2, f1_vals2, loose_vals2, strict_vals2 = [], [], [], []

for r in pred2_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    em_vals2.append(em_squad(r.prediction, refs))
    f1_vals2.append(f1_squad(r.prediction, refs))
    loose_vals2.append(em_loose_mirage(r.prediction, refs))
    strict_vals2.append(em_strict_mirage(r.prediction, refs))

best_refs2 = []
for r in pred2_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    best_refs2.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else "")

pred_texts2 = pred2_df["prediction"].astype(str).str.strip().replace("", "[NO_ANSWER]").tolist()
ref_texts2 = [str(x).strip() if str(x).strip() else "[NO_ANSWER]" for x in best_refs2]

P2, R2, F2 = bert_score(
    pred_texts2,
    ref_texts2,
    model_type=BERTSCORE_MODEL,
    lang="en",
    verbose=False,
)

metrics2 = {
    "EM": float(np.mean(em_vals2)),
    "F1": float(np.mean(f1_vals2)),
    "EM_loose": float(np.mean(loose_vals2)),
    "EM_strict": float(np.mean(strict_vals2)),
    "BERTScore_F1": float(F2.mean().item()),
    "N": int(len(pred2_df)),
}
pd.DataFrame([metrics2])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8113.99it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,0.554,0.66492,0.733,0.554,0.944369,1000


In [23]:
pred2_df.to_csv(TASK2_PRED_PATH, index=False)
pd.DataFrame([metrics2]).to_csv(TASK2_METRICS_PATH, index=False)
print(f"Saved: {TASK2_PRED_PATH}, {TASK2_METRICS_PATH}")


Saved: HW_5_task2_predictions.csv, HW_5_task2_metrics.csv


### вывод
- На той же выборке `N=1000` получены: `EM=0.554`, `F1=0.665`, `EM_loose=0.733`, `EM_strict=0.554`, `BERTScore_F1=0.944`.
- По всем метрикам Task 2 существенно лучше Task 1 (например, `EM`: `0.554` против `0.010`, `F1`: `0.665` против `0.057`).
- В oracle-режиме extractive QA на релевантном passage даёт значительно более точные ответы.


## Task 3


In [24]:
TASK3_PRED_PATH = Path("HW_5_task3_predictions.csv")
TASK3_METRICS_PATH = Path("HW_5_task3_metrics.csv")

task3_df = df.copy()
task3_df["oracle_passage"] = task3_df["oracle"].apply(
    lambda x: x.get("doc_chunk", "") if isinstance(x, dict) else ""
)
task3_df = task3_df[["query_id", "question", "answers", "oracle_passage"]]
task3_df.head(2)


,query_id,question,answers,oracle_passage
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]","Accordingly, the gymnasium became connected wi..."
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",Intercollegiate sports teams of Duke Universit...


In [25]:
def generate_openbook_answer(question: str, passage: str) -> str:
    if not isinstance(passage, str) or not passage.strip():
        return ""

    messages = [
        {
            "role": "system",
            "content": (
                "You answer factoid questions using the provided passage. "
                "Return only a short answer phrase. Do not write a full sentence."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Passage:\n{passage}\n\n"
                f"Question: {question}\n"
                "Answer with a short phrase based only on the passage."
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    ans = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip().split("\n")[0].strip()

    if "." in ans:
        ans = ans.split(".")[0].strip()

    return ans


In [29]:
rows3 = []
for r in tqdm(task3_df.itertuples(index=False), total=len(task3_df), desc="Task3 open-book LM"):
    pred = generate_openbook_answer(r.question, r.oracle_passage)
    rows3.append({
        "query_id": r.query_id,
        "question": r.question,
        "answers": r.answers,
        "oracle_passage": r.oracle_passage,
        "prediction": pred,
    })

pred3_df = pd.DataFrame(rows3)
pred3_df.head(5)


Task3 open-book LM: 100%|██████████| 1000/1000 [7:52:56<00:00, 28.38s/it]   


,query_id,question,answers,oracle_passage,prediction
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]","Accordingly, the gymnasium became connected wi...",Peripatetic school
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",Intercollegiate sports teams of Duke Universit...,"from the French ""les Diables Bleus"" or"
2,742f34d8-c3ed-49b1-acbe-455378b1f328,In what city was Joachim Olsen born?,"[Aalborg, Ålborg]",Danish politician and shot putter\nJoachim Brø...,Aalborg
3,cb036dcf-6d57-40dd-b0af-f97cf9b057fa,Who is the father of Charites?,[Zeus],Greek goddesses of grace and beauty\nIn Greek ...,Hera
4,76212448-e12d-4f2c-8bb6-d944d033af85,Who was the composer of Stormy Monday?,"[Mike Figgis, Michael Lawrence Dundus Figgis, ...",1988 crime thriller film by Mike Figgis\nStorm...,Mike Figgis


In [30]:
em_vals3, f1_vals3, loose_vals3, strict_vals3 = [], [], [], []

for r in pred3_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    em_vals3.append(em_squad(r.prediction, refs))
    f1_vals3.append(f1_squad(r.prediction, refs))
    loose_vals3.append(em_loose_mirage(r.prediction, refs))
    strict_vals3.append(em_strict_mirage(r.prediction, refs))

best_refs3 = []
for r in pred3_df.itertuples(index=False):
    refs = r.answers if isinstance(r.answers, list) else []
    best_refs3.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else "")

pred_texts3 = pred3_df["prediction"].astype(str).str.strip().replace("", "[NO_ANSWER]").tolist()
ref_texts3 = [str(x).strip() if str(x).strip() else "[NO_ANSWER]" for x in best_refs3]

P3, R3, F3 = bert_score(
    pred_texts3,
    ref_texts3,
    model_type=BERTSCORE_MODEL,
    lang="en",
    verbose=False,
)

metrics3 = {
    "EM": float(np.mean(em_vals3)),
    "F1": float(np.mean(f1_vals3)),
    "EM_loose": float(np.mean(loose_vals3)),
    "EM_strict": float(np.mean(strict_vals3)),
    "BERTScore_F1": float(F3.mean().item()),
    "N": int(len(pred3_df)),
}
pd.DataFrame([metrics3])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4015.44it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,0.55,0.693394,0.736,0.55,0.953707,1000


In [31]:
pred3_df.to_csv(TASK3_PRED_PATH, index=False)
pd.DataFrame([metrics3]).to_csv(TASK3_METRICS_PATH, index=False)
print(f"Saved: {TASK3_PRED_PATH}, {TASK3_METRICS_PATH}")


Saved: HW_5_task3_predictions.csv, HW_5_task3_metrics.csv


### вывод
- На выборке `N=1000` для Task 3 получены: `EM=0.550`, `F1=0.693`, `EM_loose=0.736`, `EM_strict=0.550`, `BERTScore_F1=0.954`.
- По сравнению с Task 2: `EM` почти одинаковый (`0.550` vs `0.554`), `F1` выше в Task 3 (`0.693` vs `0.665`), `EM_loose` немного выше (`0.736` vs `0.733`), `BERTScore_F1` выше (`0.954` vs `0.944`).
- По полученным метрикам open-book подход с instruction-tuned LM в oracle-режиме показывает сопоставимое или немного лучшее качество относительно extractive QA.


## Task 4
Use top-1 passage from two best rankers (`CE rerank (k=100)` and `Dense minilm_l6`) loaded from precomputed Colab artifacts.


In [48]:
PROJECT_ROOT = Path("/Users/maksimpiskaev/Проекты/IR")
TASK4_PRECOMP_DIR = PROJECT_ROOT / "HW_5" / "cache_colab"

TOP1_CE_PATH = TASK4_PRECOMP_DIR / "task4_top1_ce_k100_fixed1000.parquet"
TOP1_DENSE_PATH = TASK4_PRECOMP_DIR / "task4_top1_dense_minilm_sample1000.parquet"

print(TOP1_CE_PATH, TOP1_CE_PATH.exists())
print(TOP1_DENSE_PATH, TOP1_DENSE_PATH.exists())


/Users/maksimpiskaev/Проекты/IR/HW_5/cache_colab/task4_top1_ce_k100_fixed1000.parquet True
/Users/maksimpiskaev/Проекты/IR/HW_5/cache_colab/task4_top1_dense_minilm_sample1000.parquet True


In [49]:
top1_ce = pd.read_parquet(TOP1_CE_PATH)
top1_dense = pd.read_parquet(TOP1_DENSE_PATH)

for name, tdf in {"CE rerank (k=100)": top1_ce, "Dense minilm_l6": top1_dense}.items():
    print(name, "shape:", tdf.shape, "missing_doc_id:", int(tdf["doc_id"].isna().sum()))

top1_by_ranker = {
    "CE rerank (k=100)": top1_ce,
    "Dense minilm_l6": top1_dense,
}


CE rerank (k=100) shape: (1000, 4) missing_doc_id: 0
Dense minilm_l6 shape: (1000, 4) missing_doc_id: 0


In [50]:
task4_base_df = df[["query_id", "question", "answers"]].copy()
task4_base_df["query_id"] = task4_base_df["query_id"].astype(str)

def build_task4_input(base_df: pd.DataFrame, top1_df: pd.DataFrame) -> pd.DataFrame:
    m = top1_df[["query_id", "doc_id", "passage"]].copy()
    m["query_id"] = m["query_id"].astype(str)
    out = base_df.merge(m, on="query_id", how="left")
    out["passage"] = out["passage"].fillna("")
    return out

def evaluate_task4_for_ranker(ranker_name: str, top1_df: pd.DataFrame):
    task_df = build_task4_input(task4_base_df.copy(), top1_df)
    rows = []
    for r in tqdm(task_df.itertuples(index=False), total=len(task_df), desc=f"Task4 {ranker_name}"):
        pred = generate_openbook_answer(r.question, r.passage)
        rows.append({
            "ranker": ranker_name,
            "query_id": r.query_id,
            "question": r.question,
            "answers": r.answers,
            "doc_id": r.doc_id,
            "passage": r.passage,
            "prediction": pred,
        })
    pred_df = pd.DataFrame(rows)

    em_vals, f1_vals, loose_vals, strict_vals = [], [], [], []
    for r in pred_df.itertuples(index=False):
        refs = r.answers if isinstance(r.answers, list) else []
        em_vals.append(em_squad(r.prediction, refs))
        f1_vals.append(f1_squad(r.prediction, refs))
        loose_vals.append(em_loose_mirage(r.prediction, refs))
        strict_vals.append(em_strict_mirage(r.prediction, refs))

    best_refs = []
    for r in pred_df.itertuples(index=False):
        refs = r.answers if isinstance(r.answers, list) else []
        best_refs.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else "")

    pred_texts = pred_df["prediction"].astype(str).str.strip().replace("", "[NO_ANSWER]").tolist()
    ref_texts = [str(x).strip() if str(x).strip() else "[NO_ANSWER]" for x in best_refs]
    _P, _R, F = bert_score(pred_texts, ref_texts, model_type=BERTSCORE_MODEL, lang="en", verbose=False)

    metrics = {
        "ranker": ranker_name,
        "EM": float(np.mean(em_vals)),
        "F1": float(np.mean(f1_vals)),
        "EM_loose": float(np.mean(loose_vals)),
        "EM_strict": float(np.mean(strict_vals)),
        "BERTScore_F1": float(F.mean().item()),
        "N": int(len(pred_df)),
    }
    return pred_df, metrics


In [51]:
task4_pred_frames = []
task4_metrics_rows = []

for ranker_name, top1_df in top1_by_ranker.items():
    pred_df_r, metrics_r = evaluate_task4_for_ranker(ranker_name, top1_df)
    task4_pred_frames.append(pred_df_r)
    task4_metrics_rows.append(metrics_r)

pred4_df = pd.concat(task4_pred_frames, ignore_index=True)
metrics4_df = pd.DataFrame(task4_metrics_rows)
metrics4_df

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5948.63it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5595.74it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- U

,ranker,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,CE rerank (k=100),0.306,0.413083,0.445,0.306,0.914098,1000
1,Dense minilm_l6,0.316,0.417509,0.457,0.316,0.914524,1000


In [52]:
pred4_df.to_csv("HW_5_task4_predictions.csv", index=False)
metrics4_df.to_csv("HW_5_task4_metrics.csv", index=False)
print("Saved: HW_5_task4_predictions.csv, HW_5_task4_metrics.csv")


Saved: HW_5_task4_predictions.csv, HW_5_task4_metrics.csv


### вывод
- На выборке `N=1000` получены результаты: `CE rerank (k=100)` — `EM=0.306`, `F1=0.413`, `EM_loose=0.445`, `EM_strict=0.306`, `BERTScore_F1=0.914`; `Dense minilm_l6` — `EM=0.316`, `F1=0.418`, `EM_loose=0.457`, `EM_strict=0.316`, `BERTScore_F1=0.915`.
- В этом прогоне `Dense minilm_l6` немного лучше `CE rerank (k=100)` по всем пяти метрикам, но разница небольшая.
- Оба варианта top-1 контекста из retrieval заметно слабее oracle-режима Task 2/3 по точным метрикам.


## Task 5
Top-5 context for two best rankers + comparison with MIRAGE mixed context. Uses cache files when available.


In [65]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

PROJECT_ROOT = Path("/Users/maksimpiskaev/Проекты/IR")
HA4_CACHE_DIR = PROJECT_ROOT / "HW_4" / "cache"
TASK5_CACHE_DIR = PROJECT_ROOT / "HW_5" / "cache_colab"
TASK5_CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_PARQUET = TASK5_CACHE_DIR / "task5_base_sample1000.parquet"
CE_CTX_PARQUET = TASK5_CACHE_DIR / "task5_context_ce_top5_concat.parquet"
DENSE_CTX_PARQUET = TASK5_CACHE_DIR / "task5_context_dense_top5_concat.parquet"
MIXED_CTX_PARQUET = TASK5_CACHE_DIR / "task5_context_mirage_mixed_proxy.parquet"
METRICS_IMPORT = TASK5_CACHE_DIR / "HW_5_task5_metrics (1).csv"
PRED_IMPORT = TASK5_CACHE_DIR / "HW_5_task5_predictions (1).csv"


In [66]:
# Base sample
if BASE_PARQUET.exists():
    task5_base = pd.read_parquet(BASE_PARQUET)
else:
    task5_base = pd.read_pickle(PROJECT_ROOT / "HW_5" / "cache" / "mirage_sample_1000.pkl").copy()
    task5_base = task5_base[["query_id", "question", "answers", "oracle"]]
    task5_base.to_parquet(BASE_PARQUET, index=False)
task5_base["query_id"] = task5_base["query_id"].astype(str)
print("task5_base:", task5_base.shape)


task5_base: (1000, 4)


In [67]:
# Contexts with cache-first logic
if CE_CTX_PARQUET.exists() and DENSE_CTX_PARQUET.exists() and MIXED_CTX_PARQUET.exists():
    ctx_ce = pd.read_parquet(CE_CTX_PARQUET)
    ctx_dense = pd.read_parquet(DENSE_CTX_PARQUET)
    ctx_mixed = pd.read_parquet(MIXED_CTX_PARQUET)
else:
    mirage_docs_df, _, _ = pd.read_pickle(HA4_CACHE_DIR / "mirage_all.pkl")
    doc_ids = mirage_docs_df["doc_id"].astype(str).tolist()
    doc_texts = mirage_docs_df["text"].fillna("").astype(str).tolist()

    # CE top-5 recompute on fixed 1000
    bm25 = BM25Okapi([t.lower().split() for t in doc_texts])
    ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    rows=[]
    for r in tqdm(task5_base.itertuples(index=False), total=len(task5_base), desc="Task5 CE top-5"):
        qtext = str(r.question)
        bm25_scores = bm25.get_scores(qtext.lower().split())
        cand_idx = np.argsort(bm25_scores)[::-1][:100]
        pairs = [(qtext, doc_texts[i]) for i in cand_idx]
        ce_scores = ce_model.predict(pairs, batch_size=64, show_progress_bar=False)
        order = np.argsort(ce_scores)[::-1][:5]
        top5_idx = cand_idx[order]
        passages = [doc_texts[i] for i in top5_idx if doc_texts[i].strip()]
        rows.append({"query_id": str(r.query_id), "context": "\n\n".join(passages)})
    ctx_ce = pd.DataFrame(rows)

    # Dense top-5 on fixed 1000
    dense_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    q_emb = dense_model.encode(task5_base["question"].astype(str).tolist(), batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    d_emb = pd.read_pickle(HA4_CACHE_DIR / "mirage_minilm_l6_doc_emb_36950.pkl")
    if isinstance(d_emb, torch.Tensor):
        d_emb = d_emb.detach().cpu().numpy()
    d_norm = d_emb / (np.linalg.norm(d_emb, axis=1, keepdims=True) + 1e-12)
    sims = q_emb @ d_norm.T
    top5_idx = np.argpartition(-sims, kth=4, axis=1)[:, :5]
    rows=[]
    for i, qid in enumerate(task5_base["query_id"].tolist()):
        idxs = top5_idx[i]
        idxs = idxs[np.argsort(sims[i, idxs])[::-1]]
        passages = [doc_texts[j] for j in idxs if doc_texts[j].strip()]
        rows.append({"query_id": qid, "context": "\n\n".join(passages)})
    ctx_dense = pd.DataFrame(rows)

    # mixed proxy
    ctx_mixed = pd.DataFrame({
        "query_id": task5_base["query_id"].tolist(),
        "context": [str(o.get("doc_chunk", "")) if isinstance(o, dict) else "" for o in task5_base["oracle"].tolist()]
    })

    ctx_ce.to_parquet(CE_CTX_PARQUET, index=False)
    ctx_dense.to_parquet(DENSE_CTX_PARQUET, index=False)
    ctx_mixed.to_parquet(MIXED_CTX_PARQUET, index=False)

for name, cdf in {"CE_top5_concat": ctx_ce, "Dense_top5_concat": ctx_dense, "MIRAGE_mixed_context": ctx_mixed}.items():
    cdf["query_id"] = cdf["query_id"].astype(str)
    print(name, cdf.shape)


CE_top5_concat (1000, 2)
Dense_top5_concat (1000, 2)
MIRAGE_mixed_context (1000, 2)


In [68]:
def evaluate_task5_context(context_name: str, context_df: pd.DataFrame):
    eval_df = task5_base[["query_id", "question", "answers"]].copy()
    eval_df = eval_df.merge(context_df[["query_id", "context"]], on="query_id", how="left")
    eval_df["context"] = eval_df["context"].fillna("")

    rows = []
    for r in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc=f"Task5 {context_name}"):
        pred = generate_openbook_answer(r.question, r.context)
        rows.append({"context_name": context_name, "query_id": r.query_id, "question": r.question, "answers": r.answers, "context": r.context, "prediction": pred})
    pred_df = pd.DataFrame(rows)

    em_vals, f1_vals, loose_vals, strict_vals = [], [], [], []
    for r in pred_df.itertuples(index=False):
        refs = r.answers if isinstance(r.answers, list) else []
        em_vals.append(em_squad(r.prediction, refs))
        f1_vals.append(f1_squad(r.prediction, refs))
        loose_vals.append(em_loose_mirage(r.prediction, refs))
        strict_vals.append(em_strict_mirage(r.prediction, refs))

    best_refs = []
    for r in pred_df.itertuples(index=False):
        refs = r.answers if isinstance(r.answers, list) else []
        best_refs.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else "")

    pred_texts = pred_df["prediction"].astype(str).str.strip().replace("", "[NO_ANSWER]").tolist()
    ref_texts = [str(x).strip() if str(x).strip() else "[NO_ANSWER]" for x in best_refs]
    _P, _R, F = bert_score(pred_texts, ref_texts, model_type=BERTSCORE_MODEL, lang="en", verbose=False)

    metrics = {"context_name": context_name, "EM": float(np.mean(em_vals)), "F1": float(np.mean(f1_vals)), "EM_loose": float(np.mean(loose_vals)), "EM_strict": float(np.mean(strict_vals)), "BERTScore_F1": float(F.mean().item()), "N": int(len(pred_df))}
    return pred_df, metrics


In [69]:
# If imported Task 5 outputs exist, use them; otherwise run inference
if METRICS_IMPORT.exists() and PRED_IMPORT.exists():
    metrics5_df = pd.read_csv(METRICS_IMPORT)
    pred5_df = pd.read_csv(PRED_IMPORT)
    print("Loaded imported Task 5 results from cache_colab")
else:
    pred_frames = []
    metrics_rows = []
    for name, cdf in {"CE_top5_concat": ctx_ce, "Dense_top5_concat": ctx_dense, "MIRAGE_mixed_context": ctx_mixed}.items():
        p, m = evaluate_task5_context(name, cdf)
        pred_frames.append(p)
        metrics_rows.append(m)
    pred5_df = pd.concat(pred_frames, ignore_index=True)
    metrics5_df = pd.DataFrame(metrics_rows).sort_values("F1", ascending=False).reset_index(drop=True)

metrics5_df


Loaded imported Task 5 results from cache_colab


,context_name,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,MIRAGE_mixed_context,0.551,0.690703,0.728,0.551,0.953562,1000
1,Dense_top5_concat,0.363,0.475925,0.505,0.363,0.923192,1000
2,CE_top5_concat,0.232,0.322419,0.337,0.232,0.902557,1000


In [70]:
pred5_df.to_csv(PROJECT_ROOT / "HW_5" / "HW_5_task5_predictions.csv", index=False)
metrics5_df.to_csv(PROJECT_ROOT / "HW_5" / "HW_5_task5_metrics.csv", index=False)
print("Saved: HW_5/HW_5_task5_predictions.csv, HW_5/HW_5_task5_metrics.csv")


Saved: HW_5/HW_5_task5_predictions.csv, HW_5/HW_5_task5_metrics.csv


### вывод
- По результатам Task 5 на `N=1000`: `MIRAGE_mixed_context` — `EM=0.551`, `F1=0.690703`, `EM_loose=0.728`, `EM_strict=0.551`, `BERTScore_F1=0.953562`; `Dense_top5_concat` — `EM=0.363`, `F1=0.475925`, `EM_loose=0.505`, `EM_strict=0.363`, `BERTScore_F1=0.923192`; `CE_top5_concat` — `EM=0.232`, `F1=0.322419`, `EM_loose=0.337`, `EM_strict=0.232`, `BERTScore_F1=0.902557`.
- В этом прогоне ranking по качеству: `MIRAGE_mixed_context` > `Dense_top5_concat` > `CE_top5_concat` по всем пяти метрикам.
- Между двумя retrieval-вариантами `Dense_top5_concat` стабильно лучше `CE_top5_concat`; наибольший разрыв по `F1` составляет `0.153506`.


## Общий вывод по HW_5
- **Task 1 (0-shot, без контекста):** качество низкое по точным метрикам (`EM=0.010`, `F1=0.057`), при этом `BERTScore_F1=0.863` показывает частичную семантическую близость ответов.
- **Task 2 (extractive QA, oracle):** резкий рост качества (`EM=0.554`, `F1=0.665`, `BERTScore_F1=0.944`) за счёт извлечения ответа из релевантного oracle-контекста.
- **Task 3 (open-book SLM, oracle):** сопоставимо с Task 2 по `EM` и лучше по `F1/BERTScore` (`EM=0.550`, `F1=0.693`, `BERTScore_F1=0.954`).
- **Task 4 (top-1 retrieval context):** качество заметно ниже oracle-режима; в этом прогоне `Dense minilm_l6` немного лучше `CE rerank (k=100)` (`F1=0.418` vs `0.413`).
- **Task 5 (top-5 concat + mixed):** среди retrieval-вариантов лучший `Dense_top5_concat` (`F1=0.4759`), но лучший общий результат у `MIRAGE_mixed_context` (`F1=0.6907`), что подтверждает ключевую роль качества контекста.